In [0]:
# Imports

from pyspark.sql.functions import (
    col, to_timestamp, trim, upper, lower,
    current_timestamp, lit, when, expr
)
from pyspark.sql.types import IntegerType, StringType, DoubleType

In [0]:
CATALOG       = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

In [0]:
spark.sql (f"Use catalog {CATALOG}")
spark.sql (f"create schema if not exists {CATALOG}.{SILVER_SCHEMA}")
print(f"Schema ready : {CATALOG}.{SILVER_SCHEMA}")

In [0]:
def assert_unique(df, column_name, table_name): # Fail if duplicates
    total = df.count()
    distinct = df.select(column_name).distinct().count()
    if total != distinct:
        raise Exception(
            f"[QUALITY FAIL] {table_name}.{column_name} has "
            f"{total - distinct:,} duplicate values. Pipeline stopped."
        )
    print(f"{column_name} uniqueness confirmed - {total:,} rows")

In [0]:
def write_silver(df, table_name):
    target = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target)
    count = spark.table(target).count()
    print(f"Written to {target} - {count:,} rows")

In [0]:
print("Orders")
df_orders = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders") \
    .drop("_source_file", "_layer")
assert_unique(df_orders, "order_id", "orders")
write_silver(df_orders, "orders")

In [0]:
print("Order Items")
df_order_items = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.order_items") \
    .withColumn("order_item_id", col("order_item_id").cast(StringType())) \
    .drop("_source_file", "_layer")

write_silver(df_order_items, "order_items")

In [0]:
print("Order Payments")
df_order_payments = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.order_payments") \
    .drop("_source_file", "_layer")

write_silver(df_order_payments, "order_payments")


In [0]:
print("Order Reviews")
df_order_reviews = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.order_reviews") \
    .withColumn("review_score",
        expr("try_cast(review_score as int)")) \
    .withColumn("review_creation_date",
        expr("try_cast(review_creation_date as timestamp)")) \
    .withColumn("review_answer_timestamp",
        expr("try_cast(review_answer_timestamp as timestamp)")) \
    .filter(col("review_id").rlike("^[a-f0-9]{32}$")) \
    .dropDuplicates(["review_id"]) \
    .drop("_source_file", "_layer")

write_silver(df_order_reviews, "order_reviews")

In [0]:
print("Customers")
df_customers = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers") \
    .withColumn("customer_zip_code_prefix",
                col("customer_zip_code_prefix").cast(StringType())) \
    .drop("_source_file", "_layer")

assert_unique(df_customers, "customer_id", "customers")
write_silver(df_customers, "customers")


In [0]:
print("Products")
df_products = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.products") \
    .drop("_source_file", "_layer")

df_translation = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.product_category_translation") \
    .select("product_category_name", "product_category_name_english") \
    .drop("_source_file", "_layer")

df_products = df_products.join(
    df_translation,
    on="product_category_name",
    how="left" 
)

assert_unique(df_products, "product_id", "products")
write_silver(df_products, "products")

In [0]:
print("Sellers")
df_sellers = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.sellers") \
    .withColumn("seller_zip_code_prefix",
                col("seller_zip_code_prefix").cast(StringType())) \
    .drop("_source_file", "_layer")

assert_unique(df_sellers, "seller_id", "sellers")
write_silver(df_sellers, "sellers")

In [0]:
print("Geolocation")
df_geolocation = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.geolocation") \
    .withColumn("geolocation_zip_code_prefix",
                col("geolocation_zip_code_prefix").cast(StringType())) \
    .drop("_source_file", "_layer")

write_silver(df_geolocation, "geolocation")


In [0]:
print("Product Category Translation")
df_translation_silver = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.product_category_translation"
).drop("_source_file", "_layer")

write_silver(df_translation_silver, "product_category_translation")

In [0]:
SILVER_TABLES = [
    "orders", "order_items", "order_payments", "order_reviews",
    "customers", "products", "sellers", "geolocation",
    "product_category_translation"
]

print(f"SILVER LAYER — ROW COUNT SUMMARY")

for table in SILVER_TABLES:
    target = f"{CATALOG}.{SILVER_SCHEMA}.{table}"
    try:
        count = spark.table(target).count()
        print(f"  {table:<40} {count:>10,} rows")
    except Exception as e:
        print(f"  {table:<40} ERROR: {e}")

print("Silver layer completed.")